# CA noise robustness: contemporaneous return variant

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


**Defining the utils**

**COMMENT: restart after having run this cell, so it is in memory**

**Description**

1. Change the results path for the number hidden layer in each model, iteratively : results_CA_0, results_CA_1, results_CA_2, results_CA_0
2. Run the code that defines the desired model with the number of hidden layers
3. Run the codes that preprocess the data for the models.
4. Run the code for the cross-validation, which makes prediction on the validation set. The performance evaluation metrcs are saved in the results path.Remember to change the results path for the cross-validation.
5. Run the the code that predicts the desired model with the desired number of hidden layers. remember to change the results path for the out-of-sample  predictions on the test set. This code makes predictions for each latent latent factor across a time-series split, and the predicted results are saved.

**sliding window with validation**

In [ ]:
%%writefile utils.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
__author__ = 'Hans Andrias Gregoriussen'

import numpy as np

np.random.seed(42)


def format_time(t):
    """Return a formatted time string 'HH:MM:SS
    based on a numeric time() value"""
    m, s = divmod(t, 60)
    h, m = divmod(m, 60)
    return f'{h:0>2.0f}:{m:0>2.0f}:{s:0>2.0f}'

class MultipleTimeSeriesCV:
    """Generates tuples of train_idx, val_idx and test_idx pairs
    Assumes the MultiIndex contains levels 'symbol' and 'date'
    purges overlapping outcomes"""

    def __init__(self,
                 n_splits=3,
                 train_period_length=126,
                 test_period_length=21,
                 val_period_length=21,
                 lookahead=None,
                 date_idx='date',
                 shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.val_length = val_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle
        self.date_idx = date_idx





    def split(self, X, y=None, groups=None):
        unique_dates = X.index.get_level_values(self.date_idx).unique()
        days = sorted(unique_dates, reverse=True)
        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            val_end_idx = test_start_idx + self.lookahead
            val_start_idx = val_end_idx + self.val_length + self.lookahead
            train_end_idx = val_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead
            split_idx.append([train_start_idx, train_end_idx,
                              val_start_idx, val_end_idx,
                              test_start_idx, test_end_idx])

        dates = X.reset_index()[[self.date_idx]]
        for train_start, train_end, val_start, val_end, test_start, test_end in split_idx:

            train_idx = dates[(dates[self.date_idx] > days[train_start])
                              & (dates[self.date_idx] <= days[train_end])].index
            val_idx = dates[(dates[self.date_idx] > days[val_start])
                              & (dates[self.date_idx] <= days[val_end])].index
            test_idx = dates[(dates[self.date_idx] > days[test_start])
                             & (dates[self.date_idx] <= days[test_end])].index
            if self.shuffle:
                np.random.shuffle(list(train_idx))
            yield train_idx.to_numpy(),val_idx.to_numpy(), test_idx.to_numpy()

    def get_n_splits(self, X, y, groups=None):
        return self.n_splits



class DateTimeSeriesCV:
    """Generates tuples of train_idx,val_idx and test_idx pairs
    Assumes the index is of datetime type
    purges overlapping outcomes"""

    def __init__(self, n_splits=3, train_period_length=126, val_period_length=21, test_period_length=21, lookahead=None, shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.val_length = val_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle

    def split(self, X, y=None, groups=None):
        unique_dates = X.index.unique()
        days = sorted(unique_dates, reverse=True)

        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            val_end_idx = test_start_idx + self.lookahead
            val_start_idx = val_end_idx + self.val_length + self.lookahead
            train_end_idx = val_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead

            if train_start_idx >= len(days):
                break  # Stop if we've gone beyond the dataset

            split_idx.append([train_start_idx, train_end_idx, val_start_idx, val_end_idx, test_start_idx, test_end_idx])

        for train_start, train_end, val_start, val_end, test_start, test_end in split_idx:
            train_idx = X.index[(X.index > days[train_start]) & (X.index <= days[train_end])]
            val_idx = X.index[(X.index > days[val_start]) & (X.index <= days[val_end])]
            test_idx = X.index[(X.index > days[test_start]) & (X.index <= days[test_end])]

            if self.shuffle:
                train_idx = np.random.permutation(train_idx)

            yield train_idx, val_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


In [ ]:
!pip install tables

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
from time import time
from pathlib import Path
from itertools import product
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import time
import random
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dot, Reshape, BatchNormalization, Masking
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import quantile_transform
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

from scipy.stats import spearmanr

from utils import format_time, MultipleTimeSeriesCV
from utils import format_time, DateTimeSeriesCV

from numpy.linalg import inv
from google.colab import drive




**Setting up TPU**

In [ ]:
# # Set up the TPU
# try:
#     tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
#     print('Running on TPU ', tpu.master())

#     # Always connect to the TPU cluster and initialize the TPU system
#     tf.config.experimental_connect_to_cluster(tpu)
#     tf.tpu.experimental.initialize_tpu_system(tpu)

#     strategy = tf.distribute.TPUStrategy(tpu)
#     print("All TPU devices: ", tf.config.experimental.list_logical_devices('TPU'))
# except ValueError:
#     raise BaseException('ERROR: Not connected to a TPU runtime; select TPU from the "Runtime > Change runtime type" menu.')



**defining my results path**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/Robustness_check')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)


In [ ]:

# ###########################
# ## MOUNT TO GOOGLE DRIVE ##
# ###########################

# # Mount Google Drive
# drive.mount('/content/drive')

# ################################
# ## LOAD DATA FOR BETA NETWORK ##
# ################################


# file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'

# # Make sure merged_df1 is a DataFrame and not a function or any other type.
# merged_df_padded = pd.read_parquet(file_path)

# # Assuming merged_df_padded is already defined
# # Ensure the date level of the MultiIndex is of datetime type
# merged_df_padded.index = merged_df_padded.index.set_levels(pd.to_datetime(merged_df_padded.index.levels[0]), level=0)

# # Filter the DataFrame to include only rows from 1979 and onwards
# merged_df_padded = merged_df_padded.loc[merged_df_padded.index.get_level_values('date') >= '1979-01-01']

# print(merged_df_padded)
# print(merged_df_padded.index)
# print(merged_df_padded.shape)
# print(type(merged_df_padded))


# #################################
# ## LOAD DATA FOR FACTOR NETWORK##
# #################################

# # Define the file path
# file_path = '/content/drive/My Drive/Tidy Finance ML dataset/x_t_preprocessed.parquet'

# # Load the Parquet file into a DataFrame
# portfolio_rets_df = pd.read_parquet(file_path)



# # Assuming portfolio_rets_df is already defined and loaded
# # Ensure the date index is of datetime type
# portfolio_rets_df.index = pd.to_datetime(portfolio_rets_df.index)

# # Filter the DataFrame to include only rows from 1979 and onwards
# portfolio_rets_df = portfolio_rets_df.loc[portfolio_rets_df.index >= '79-01-01']

# print(portfolio_rets_df)
# print(portfolio_rets_df.index)
# print(portfolio_rets_df.shape)
# print(type(portfolio_rets_df))




In [ ]:
# import pandas as pd
# import numpy as np

# # Mount Google Drive
# drive.mount('/content/drive')

# # Load data for beta network
# beta_file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'
# merged_df_padded = pd.read_parquet(beta_file_path)
# merged_df_padded.index = merged_df_padded.index.set_levels(pd.to_datetime(merged_df_padded.index.levels[0]), level=0)
# merged_df_padded = merged_df_padded.loc[merged_df_padded.index.get_level_values('date') >= '1970-01-01']

# # Remove rows with any -2 values
# merged_df_cleaned = merged_df_padded[~(merged_df_padded == -2).any(axis=1)]




# print(merged_df_cleaned)
# print(merged_df_cleaned.index)
# print(merged_df_cleaned.shape)
# print(type(merged_df_cleaned))


In [ ]:
# import numpy as np
# import pandas as pd
# from tqdm import tqdm

# # Define the list of characteristic codes
# annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
#                 "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
#                 "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
#                 "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
#                 "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
#                 "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
#                 "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
# quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
#                    "stdacc", "stdcf"]
# monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
#                  "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
#                  "zerotrade", "equal_weighted_portfolio"]
# # Combine all codes
# all_codes = annual_codes + quarterly_codes + monthly_codes
# characteristics = [f'characteristic_{code}' for code in all_codes]

# # Assuming merged_df_padded is already defined and loaded

# # Replace -2 with NaN to correctly handle missing values
# merged_df_padded.replace(-2, np.nan, inplace=True)

# # Parameters for noise generation
# unique_permnos = merged_df_padded.index.get_level_values('permno').unique()
# unique_dates = merged_df_padded.index.get_level_values('date').unique()
# N = len(unique_permnos)  # Number of unique permnos
# T = len(unique_dates)  # Number of time periods
# Pc = 5  # Number of noise characteristics

# # Calculate the standard deviation of the specified columns ignoring NaNs
# characteristics_std = merged_df_padded[characteristics].std(skipna=True).mean()

# # Simulate noise characteristics
# rho = np.random.uniform(0.9, 1, Pc)
# c = np.zeros((N, Pc, T))

# # Generate the noise characteristics
# for t in tqdm(range(1, T), desc="Generating noise characteristics"):
#     c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

# # Normalize noise characteristics
# c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

# # Scale noise characteristics
# c_scaled = c_normalized * characteristics_std

# # Check the shape of c_scaled
# print(c_scaled.shape)  # (N, Pc, T)

# # Reshape scaled characteristics to fit the DataFrame structure
# c_scaled = c_scaled.transpose(2, 0, 1)  # Transpose to (T, N, Pc)
# c_scaled = c_scaled.reshape(T * N, Pc)  # Flatten to (T*N, Pc)

# # Create a new MultiIndex to match the merged_df_padded index
# dates = np.repeat(unique_dates, N)
# permnos = np.tile(unique_permnos, T)

# # Verify the lengths of dates and permnos arrays
# print(f'Number of dates: {len(dates)}')  # Should be T * N
# print(f'Number of permnos: {len(permnos)}')  # Should be T * N

# # Create the new MultiIndex
# new_index = pd.MultiIndex.from_arrays([dates, permnos], names=['date', 'permno'])

# # Create the DataFrame
# noise_features_df = pd.DataFrame(c_scaled, index=new_index, columns=[f'noise_{i}' for i in range(Pc)])

# # Ensure the new index matches the original DataFrame index
# noise_features_df = noise_features_df.loc[merged_df_padded.index]

# # Add noise features to the original characteristics
# beta_with_noise = pd.concat([merged_df_padded, noise_features_df], axis=1)

# # Replace NaNs back with -2 to maintain the same structure as the original data
# beta_with_noise.fillna(-2, inplace=True)

# # Print to check the DataFrame
# print(beta_with_noise.head())
# print(beta_with_noise.shape)
# print(type(beta_with_noise))


In [ ]:
# import numpy as np
# import pandas as pd

# # Assuming portfolio_rets_df is already defined and loaded

# # Define the list of characteristic codes (same as before)
# annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
#                 "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
#                 "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
#                 "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
#                 "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
#                 "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
#                 "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
# quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
#                    "stdacc", "stdcf"]
# monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
#                  "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
#                  "zerotrade", "equal_weighted_portfolio"]
# # Combine all codes
# all_codes = annual_codes + quarterly_codes + monthly_codes
# characteristics = [f'characteristic_{code}' for code in all_codes]

# # Parameters for noise generation
# unique_dates_portfolio = portfolio_rets_df.index.unique()
# T_portfolio = len(unique_dates_portfolio)  # Number of time periods for portfolio returns
# Pc_portfolio = 5  # Number of noise portfolios

# # Calculate the standard deviation of the specified columns
# characteristics_std_portfolio = portfolio_rets_df[characteristics].std().mean()

# # Simulate noise portfolios
# rho_portfolio = np.random.uniform(0.9, 1, Pc_portfolio)
# c_portfolio = np.zeros((T_portfolio, Pc_portfolio))

# # Generate the noise portfolios
# for t in range(1, T_portfolio):
#     c_portfolio[t, :] = rho_portfolio * c_portfolio[t - 1, :] + np.random.normal(0, 1, Pc_portfolio)

# # Normalize noise portfolios
# c_normalized_portfolio = 2 / (T_portfolio + 1) * (c_portfolio.argsort(axis=0).argsort(axis=0) - 1) - 1

# # Scale noise portfolios
# c_scaled_portfolio = c_normalized_portfolio * characteristics_std_portfolio

# # Check the shape of c_scaled_portfolio
# print(c_scaled_portfolio.shape)  # (T_portfolio, Pc_portfolio)

# # Create a DataFrame for noise portfolios with the same index as portfolio_rets_df
# noise_portfolios_df = pd.DataFrame(c_scaled_portfolio, index=portfolio_rets_df.index, columns=[f'noise_{i}' for i in range(Pc_portfolio)])

# # Add noise portfolios to the original portfolio returns
# portfolios_with_noise = pd.concat([portfolio_rets_df, noise_portfolios_df], axis=1)

# # Print to check the DataFrame
# print(portfolios_with_noise.head())
# print(portfolios_with_noise.shape)
# print(type(portfolios_with_noise))

# # Fill missing values
# final_df = final_df.fillna(-2)

# print(final_df.index)
# print(final_df.shape)
# print(final_df)



In [ ]:

# # Define the list of characteristic codes
# annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
#                 "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
#                 "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
#                 "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
#                 "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
#                 "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
#                 "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
# quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
#                    "stdacc", "stdcf"]
# monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
#                  "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
#                  "zerotrade", "equal_weighted_portfolio"]
# # Combine all codes
# all_codes = annual_codes + quarterly_codes + monthly_codes
# characteristics = [f'characteristic_{code}' for code in all_codes]

# # # Select the required columns including the noise characteristics
# # columns_to_select = characteristics + [f'noise_{i}' for i in range(Pc)] + ["ret_excess"]
# # characteristics_data = beta_with_noise[columns_to_select]



# # # # Select the required columns including the noise characteristics
# # # characteristics = list(merged_df_cleaned.columns)
# # # columns_to_select = characteristics + [f'noise_{i}' for i in range(Pc)] + ["ret_excess"]
# # # characteristics_data = beta_with_noise[columns_to_select]

# # # Function to calculate x_t for a given month using Ridge Regression
# # def calculate_xt(df, date, alpha=0):
# #     # Filter DataFrame for the given date
# #     df_month = df.loc[date]

# #     # Separate returns and characteristics
# #     r_t = df_month['ret_excess'].values
# #     characteristic_columns = df_month.drop(columns=['ret_excess']).columns
# #     Z_t_minus_1 = df_month.drop(columns=['ret_excess']).values

# #     # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
# #     Z_t_minus_1_T = Z_t_minus_1.T
# #     Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
# #     Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

# #     Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)

# #     # Calculate x_t
# #     x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t

# #     return x_t, characteristic_columns

# # # Calculate x_t for all unique dates in the dataset
# # def calculate_xt_all_dates(df, alpha=1e-6):
# #     xt_results = {}
# #     characteristic_columns = None

# #     unique_dates = df.index.get_level_values('date').unique()
# #     for date in unique_dates:
# #         x_t, characteristic_columns = calculate_xt(df, date, alpha)
# #         xt_results[date] = x_t

# #     # Convert dictionary to DataFrame
# #     xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

# #     # Assign original characteristic names as column names
# #     if characteristic_columns is not None:
# #         xt_df.columns = characteristic_columns

# #     # Preserve the index name
# #     xt_df.index.name = 'date'

# #     return xt_df

# # # Example usage
# # xt_all_dates = calculate_xt_all_dates(characteristics_data)


**MODEL PARAMETER SPECIFICATIONS**

In [ ]:
import pandas as pd

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define the path to load the files
load_path = '/content/drive/My Drive/Tidy Finance ML dataset/'

# Load final_df from the HDF5 file
beta_with_noise = pd.read_hdf(f'{load_path}/final_df.h5', key='final_df')

# Load xt_all_dates from the HDF5 file
portfolios_with_noise = pd.read_hdf(f'{load_path}/xt_all_dates.h5', key='xt_all_dates')

# Verify the loaded data
print(beta_with_noise.info())
print(beta_with_noise.head())

print(portfolios_with_noise.info())
print(portfolios_with_noise.head())


In [ ]:
Pc=5
# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade", "equal_weighted_portfolio"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Select the required columns including the noise characteristics
columns_to_select = characteristics + [f'noise_{i}' for i in range(Pc)]

narratives = columns_to_select
print(narratives)


n_factors = 6
n_narratives = len(columns_to_select)
n_tickers = len(beta_with_noise.index.unique('permno'))

print(n_narratives)
print(n_tickers)



**Converting data to float16 for efficiency and saving files**

In [ ]:
# import pandas as pd


# # Define the columns to select
# columns_to_select = narratives + ["ret_excess", "fwd_ret_excess"]

# # Convert specified columns to float16 directly in final_df
# for column in columns_to_select:
#     beta_with_noise[column] = beta_with_noise[column].astype('float16')

# print(beta_with_noise)


# # Define the columns to select
# columns_to_select = narratives

# # Convert specified columns to float16 directly in xt_all_dates
# for column in columns_to_select:
#     portfolios_with_noise[column] = portfolios_with_noise[column].astype('float16')

# print(portfolios_with_noise)



In [ ]:
# import pandas as pd

# # Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# # Define the path to save the files
# save_path = '/content/drive/My Drive/Tidy Finance ML dataset/'

# # Define the columns to select
# columns_to_select = narratives + ["ret_excess", "fwd_ret_excess"]

# # Convert specified columns to float16 directly in final_df
# for column in columns_to_select:
#     beta_with_noise[column] = beta_with_noise[column].astype('float16')

# print(beta_with_noise)
# portfolios_with_noise

# # Save final_df as an HDF5 file
# beta_with_noise.to_hdf(f'{save_path}/final_df.h5', key='final_df', mode='w')

# # Define the columns to select
# columns_to_select = narratives

# # Convert specified columns to float16 directly in xt_all_dates
# for column in columns_to_select:
#     portfolios_with_noise[column] = portfolios_with_noise[column].astype('float16')

# print(portfolios_with_noise)

# # Save xt_all_dates as an HDF5 file
# portfolios_with_noise.to_hdf(f'{save_path}/xt_all_dates.h5', key='xt_all_dates', mode='w')


**WORKS2**

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot, Masking
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import spearmanr
import time
from itertools import product
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm
# Define a custom loss function
def custom_loss(mask_value=-2):
    def loss(y_true, y_pred):
        mask = tf.not_equal(y_true, mask_value)
        mask = tf.cast(mask, dtype=tf.float32)
        mse = tf.square(y_pred - y_true)
        mse *= mask  # Apply the mask to the MSE
        return tf.reduce_sum(mse) / tf.reduce_sum(mask)  # Normalize by the number of valid values
    return loss

# Function to create the model
def make_model(n_tickers, n_narratives, n_factors=6, l1_reg=0.01, num_hidden_layers=0, return_components=False, mask_value=-2):
    input_beta = Input((n_tickers, n_narratives), name='input_beta')
    input_factor = Input((n_narratives,), name='input_factor')

    # Apply Masking layer
    x = Masking(mask_value=mask_value, name='masking_layer')(input_beta)

    # Add hidden layers conditionally
    if num_hidden_layers >= 1:
        hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_1')(hidden_layer_1)
    if num_hidden_layers >= 2:
        hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_2')(hidden_layer_2)
    if num_hidden_layers >= 3:
        hidden_layer_3 = Dense(units=8, activation='relu', name='hidden_layer_3', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_3')(hidden_layer_3)

    # Output beta layer
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(x)

    # Output factor layer
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Combine beta and factor through a dot product
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the model with a custom loss function
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss=custom_loss(mask_value=mask_value), optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both models
    else:
        return model  # Return only the main model

# Function to get train, validation, and test data combined
def get_train_valid_data_combined(merged_data, portfolio_data, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio):
    # Process merged_df_padded data
    train_merged, val_merged, test_merged = merged_data.iloc[train_idx_merged], merged_data.iloc[val_idx_merged], merged_data.iloc[test_idx_merged]

    X1_train = train_merged.loc[:, columns_to_select].values.reshape(-1, n_tickers, n_narratives)
    X1_val = val_merged.loc[:, columns_to_select].values.reshape(-1, n_tickers, n_narratives)
    X1_test = test_merged.loc[:, columns_to_select].values.reshape(-1, n_tickers, n_narratives)

    y_train = train_merged.ret_excess.unstack('permno')
    y_val = val_merged.ret_excess.unstack('permno')
    y_test = test_merged.ret_excess.unstack('permno')

    # Process loaded_portfolio_rets data
    X2_train = portfolio_data.loc[train_dates_portfolio]
    X2_val = portfolio_data.loc[val_dates_portfolio]
    X2_test = portfolio_data.loc[test_dates_portfolio]

    return X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test

# Function to compute variable importance
def compute_permutation_variable_importance(model, X1, X2, y):
    # Get the baseline performance
    preds = model.predict([X1, X2])
    baseline_r2 = r2_score(y.values.flatten(), preds.flatten())

    importances = []

    # Iterate over each characteristic
    for i in range(X1.shape[2]):
        X1_copy = X1.copy()
        np.random.shuffle(X1_copy[:, :, i])  # Shuffle the i-th characteristic

        preds_permuted = model.predict([X1_copy, X2])
        permuted_r2 = r2_score(y.values.flatten(), preds_permuted.flatten())

        importance = baseline_r2 - permuted_r2
        importances.append(importance)

    return importances

# Seed and strategy setup
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Create a function to convert your data into tf.data.Dataset
def make_dataset(X1, X2, y, batch_size, is_training=True):
    dataset = tf.data.Dataset.from_tensor_slices(((X1, X2), y))
    if is_training:
        dataset = dataset.shuffle(buffer_size=len(X1), seed=SEED)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
    return dataset

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

cols = ['num_hidden_layers', 'n_factors', 'l1_reg', 'fold', 'r2_score']

max_epochs = 100
factor_opts = [5]
l1_reg_opts = [1e-9, 1e-7, 1e-5, 1e-3]
#
hidden_layer_opts = [0, 1, 2, 3]
#
param_grid = list(product(hidden_layer_opts, factor_opts))
units = 32
batch_size = 240

YEAR = 12
cv_merged = MultipleTimeSeriesCV(n_splits=20, train_period_length=20* YEAR, val_period_length=1 * YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)
cv_portfolio = DateTimeSeriesCV(n_splits=20, train_period_length=20* YEAR, val_period_length=1 *YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

start = time.time()

# Create the HDF5 file initially in 'table' format
with pd.HDFStore(str(results_path / 'scores.h5'), mode='w', complevel=9, complib='blosc:zstd') as store:
    store.close()

# Define strategy for distributed training
strategy = tf.distribute.MirroredStrategy()

# Initialize DataFrame to store the tuning results
tuning_results = pd.DataFrame(columns=cols)

# Directory to save the model weights
weights_dir = Path('model_weights')
weights_dir.mkdir(exist_ok=True)

# First, iterate over parameter grid to find the best l1_reg for each num_hidden_layers and n_factors combination
for num_hidden_layers, n_factors in tqdm(param_grid):
    for l1_reg in tqdm(l1_reg_opts):
        fold_r2_scores = []
        fold_results = []  # Temporary list to store results for this iteration
        with strategy.scope():
             main_model, factor_network_model = make_model(n_tickers, n_factors=n_factors, n_narratives=n_narratives, num_hidden_layers=num_hidden_layers, return_components=True, l1_reg=l1_reg, mask_value=-2)

        for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(beta_with_noise), cv_portfolio.split(portfolios_with_noise))):

            X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(beta_with_noise, portfolios_with_noise, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

            # Create datasets
            train_dataset = make_dataset(X1_train, X2_train, y_train, batch_size).repeat()
            val_dataset = make_dataset(X1_val, X2_val, y_val, batch_size, is_training=False).repeat()

            # Distribute the datasets
            train_dist_dataset = strategy.experimental_distribute_dataset(train_dataset)
            val_dist_dataset = strategy.experimental_distribute_dataset(val_dataset)

            # Calculate the number of steps per epoch
            steps_per_epoch = max(len(X1_train) // batch_size, 1)
            validation_steps = max(len(X1_val) // batch_size, 1)

            # Training the whole model
            main_model.fit(train_dist_dataset,
                           validation_data=val_dist_dataset,
                           epochs=max_epochs,
                           verbose=0,
                           steps_per_epoch=steps_per_epoch,
                           validation_steps=validation_steps,
                           callbacks=[early_stopping])

            # Directly call the predict method on the model
            val_preds = main_model.predict(val_dist_dataset, steps=validation_steps)
            result = pd.DataFrame({'y_pred': val_preds.flatten(), 'y_true': y_val.stack().values}, index=y_val.stack().index).replace(-2, np.nan).dropna()

            # Calculate R²
            r2 = r2_score(result.y_true, result.y_pred)
            fold_r2_scores.append(r2)
            print(f'Model: CA {num_hidden_layers}|No. Factors : {n_factors}|lasso: {l1_reg}|FOLD :{fold}|VAL R^2: {r2}')

            # Append metrics to the list
            fold_results.append({
                'num_hidden_layers': num_hidden_layers,
                'n_factors': n_factors,
                'l1_reg': l1_reg,
                'fold': fold,
                'r2_score': r2
            })

        # Concatenate fold results to the main DataFrame
        tuning_results = pd.concat([tuning_results, pd.DataFrame(fold_results)], ignore_index=True)

        #save model weights
        weights_path = weights_dir / f'weights_{int(num_hidden_layers)}_{int(n_factors)}_{l1_reg}.h5'
        main_model.save_weights(weights_path)

# Ensure all columns have consistent data types before saving
tuning_results['num_hidden_layers'] = tuning_results['num_hidden_layers'].astype(int)
tuning_results['n_factors'] = tuning_results['n_factors'].astype(int)
tuning_results['fold'] = tuning_results['fold'].astype(int)
tuning_results['r2_score'] = tuning_results['r2_score'].astype(float)

# Save the tuning results DataFrame
tuning_results.to_hdf(str(results_path / 'tuning_results.h5'), key='tuning_results', format='table')

# Load the tuning results DataFrame
tuning_results = pd.read_hdf(str(results_path / 'tuning_results.h5'), key='tuning_results')

# Calculate the average R² score for each combination of num_hidden_layers, n_factors, and l1_reg
avg_r2_scores = tuning_results.groupby(['num_hidden_layers', 'n_factors', 'l1_reg'])['r2_score'].mean().reset_index()

# Find the best l1_reg for each combination of num_hidden_layers and n_factors based on the highest average R² score
best_l1_reg_per_layer_factor = avg_r2_scores.loc[avg_r2_scores.groupby(['num_hidden_layers', 'n_factors'])['r2_score'].idxmax()]

# Reinitialize strategy for distributed training
strategy = tf.distribute.MirroredStrategy()

# Initialize lists to store final test predictions and importances
test_predictions_list = []
fold_importances_list = []

start = time.time()

# Final training and evaluation using the best l1_reg value for each num_hidden_layers and n_factors
for _, row in tqdm(best_l1_reg_per_layer_factor.iterrows()):
    num_hidden_layers = row['num_hidden_layers']
    n_factors = row['n_factors']
    best_l1_reg = row['l1_reg']

    # Reload the best model weights
    weights_path = weights_dir / f'weights_{int(num_hidden_layers)}_{int(n_factors)}_{best_l1_reg}.h5'
    with strategy.scope():
        main_model, factor_network_model = make_model(n_tickers, n_factors=n_factors, n_narratives=n_narratives, num_hidden_layers=num_hidden_layers, return_components=True, l1_reg=best_l1_reg, mask_value=-2)
        main_model.load_weights(weights_path)

    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(beta_with_noise), cv_portfolio.split(portfolios_with_noise))):
        fold_scores = []
        fold_val_predictions_list = []

        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(beta_with_noise, portfolios_with_noise, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

        # Create datasets
        train_dataset = make_dataset(X1_train, X2_train, y_train, batch_size).repeat()
        val_dataset = make_dataset(X1_val, X2_val, y_val, batch_size, is_training=False).repeat()
        test_dataset = make_dataset(X1_test, X2_test, y_test, batch_size, is_training=False).repeat()

        # Distribute the datasets
        train_dist_dataset = strategy.experimental_distribute_dataset(train_dataset)
        val_dist_dataset = strategy.experimental_distribute_dataset(val_dataset)

        # Calculate the number of steps per epoch
        steps_per_epoch = max(len(X1_train) // batch_size, 1)
        validation_steps = max(len(X1_val) // batch_size, 1)
        test_steps = max(len(X1_test) // batch_size, 1)

        # Training the whole model
        main_model.fit(train_dist_dataset,
                       validation_data=val_dist_dataset,
                       epochs=max_epochs,
                       verbose=0,
                       steps_per_epoch=steps_per_epoch,
                       validation_steps=validation_steps,
                       callbacks=[early_stopping])

        # Evaluate the model on validation set
        val_preds = main_model.predict(val_dist_dataset, steps=validation_steps)
        result = pd.DataFrame({'y_pred': val_preds.flatten(), 'y_true': y_val.stack().values}, index=y_val.stack().index).replace(-2, np.nan).dropna()

        # Calculate R²
        r2 = r2_score(result.y_true, result.y_pred)
        print(f'Time :{time.time()-start:.2f}|num_hidden_layers: {num_hidden_layers}|n_factors: {n_factors}|lasso: {best_l1_reg}|FOLD{fold}|VAL R^2: {r2}')
        fold_scores.append([num_hidden_layers, n_factors, best_l1_reg, fold, r2])

        # Save validation predictions for this fold
        fold_val_predictions_list.append(result)

        # Save validation predictions for this fold
        fold_val_predictions_df = pd.concat(fold_val_predictions_list)
        fold_val_predictions_df.to_hdf(str(results_path / 'validation_predictions.h5'), key=f'fold_{fold}', append=True, format='table')

        fold_scores_df = pd.DataFrame(fold_scores, columns=cols)
        fold_scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'fold_{fold}', append=True, format='table')

        # Evaluate the model on test set
        test_preds = main_model.predict(test_dataset, steps=test_steps)
        test_result = pd.DataFrame({'y_pred': test_preds.flatten(), 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()

        # Save test predictions for this fold
        test_predictions_list.append(test_result)

        # Compute variable importance using the best model for this fold and test set
        importances = compute_permutation_variable_importance(main_model, X1_test, X2_test, y_test)
        fold_importances_list.append(importances)

# Save all test predictions
all_test_predictions_df = pd.concat(test_predictions_list)
all_test_predictions_df.to_hdf(str(results_path / 'test_predictions.h5'), key='all_folds', append=True, format='table')

# Calculate average variable importances across all folds
avg_importances = np.mean(fold_importances_list, axis=0)
importances_df = pd.DataFrame(avg_importances, index=columns_to_select, columns=['importance'])
importances_df['importance_normalized'] = importances_df['importance'] / importances_df['importance'].sum()
importances_df = importances_df.reset_index().rename(columns={'index': 'variable'})
importances_df['variable'] = importances_df['variable'].str.replace('characteristic_', '')
importances_df.to_hdf(str(results_path / 'variable_importance.h5'), key='all_folds', append=True, format='table')

# Plot the top 20 variable importances
top_20_importances = importances_df.sort_values('importance_normalized', ascending=False).head(20)
plt.figure(figsize=(12, 10))
sns.barplot(x=top_20_importances['importance_normalized'], y=top_20_importances['variable'])
plt.xlabel('Normalized Importance')
plt.ylabel('Variables')
plt.title('Top 20 Normalized Variable Importances')
plt.show()

print("Predictions and variable importance computation complete.")


In [ ]:
weights_path